In [1]:
!pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 55.6 MB/s eta 0:00:00


In [2]:
!pip install pymupdf

import os
import fitz
import pandas as pd
import nltk
import re
from multiprocessing import Pool, cpu_count
from nltk.corpus import stopwords
import warnings

warnings.filterwarnings("ignore")

nltk.download("stopwords", quiet=True)
stop_words = set(stopwords.words("english"))

reports_folder = "/kaggle/input/datasets/meghanakadari/indian-companies-annual-reports-fy-202223/Annual_Report_23/annual_reports_2023"

sections = [
    "general disclosures",
    "management and process disclosures",
    "principle wise performance",
    "environment",
    "social",
    "governance"
]


def process_pdf(file):

    company = file.replace(".pdf", "")
    path = os.path.join(reports_folder, file)

    text = ""

    try:
        doc = fitz.open(path)

        # read only first 60 pages (BRSR usually early in report)
        for page in doc[:60]:
            text += page.get_text()

    except:
        return None

    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)

    words = text.split()

    total_words = len(words)

    words_no_stop = [w for w in words if w not in stop_words]

    total_words_no_stop = len(words_no_stop)

    results = []

    results.append({
        "Company": company,
        "Section": "TOTAL_REPORT",
        "Word_Count": total_words,
        "Word_Count_No_Stopwords": total_words_no_stop
    })

    for section in sections:

        if section in text:

            section_text = text.split(section, 1)[1]

            words = section_text.split()

            wc = len(words)

            words_no_stop = [w for w in words if w not in stop_words]

            wc_no_stop = len(words_no_stop)

            results.append({
                "Company": company,
                "Section": section,
                "Word_Count": wc,
                "Word_Count_No_Stopwords": wc_no_stop
            })

    return results


if __name__ == "__main__":

    files = [f for f in os.listdir(reports_folder) if f.endswith(".pdf")]

    print("Total reports:", len(files))

    with Pool(cpu_count()) as pool:
        results = pool.map(process_pdf, files)

    data = []

    for r in results:
        if r:
            data.extend(r)

    df = pd.DataFrame(data)

    section_stats = df.groupby("Section")[["Word_Count","Word_Count_No_Stopwords"]].sum()

    company_stats = df[df["Section"]=="TOTAL_REPORT"][["Company","Word_Count","Word_Count_No_Stopwords"]]

    section_presence = pd.crosstab(df["Company"], df["Section"])

    df.to_csv("detailed_word_counts_Ann_22_23.csv", index=False)
    section_stats.to_csv("section_summary_Ann_22_23.csv")
    company_stats.to_csv("company_summary_Ann_22_23.csv")
    section_presence.to_csv("section_presence_Ann_22_23.csv")

    print("EDA Completed")

Total reports: 1394
MuPDF error: library error: FT_New_Memory_Face(UJALIE+Helvetica-Bold): unknown file format

MuPDF error: library error: FT_New_Memory_Face(UJALIE+Helvetica): unknown file format

MuPDF error: argument error: cannot create appearance stream for  widgets

MuPDF error: argument error: cannot create appearance stream for  widgets

MuPDF error: argument error: cannot create appearance stream for  widgets

MuPDF error: argument error: cannot create appearance stream for  widgets

MuPDF error: argument error: cannot create appearance stream for  widgets

MuPDF error: argument error: cannot create appearance stream for  widgets

MuPDF error: argument error: cannot create appearance stream for  widgets

MuPDF error: argument error: cannot create appearance stream for  widgets

MuPDF error: argument error: cannot create appearance stream for  widgets

MuPDF error: argument error: cannot create appearance stream for  widgets

MuPDF error: argument error: cannot create appearan

In [3]:
import os
import fitz
import pandas as pd
import re
from multiprocessing import Pool, cpu_count

reports_folder = "/kaggle/input/datasets/meghanakadari/indian-companies-annual-reports-fy-202223/Annual_Report_23/annual_reports_2023"

# Sector keyword dictionary
sector_keywords = {
    "Information Technology": ["software", "technology", "it services", "digital"],
    "Financial Services": ["bank", "finance", "financial services", "nbfc", "insurance"],
    "Energy": ["oil", "gas", "energy", "petroleum"],
    "Healthcare": ["pharma", "pharmaceutical", "healthcare", "biotech"],
    "Consumer Goods": ["consumer goods", "fmcg", "retail", "food"],
    "Industrials": ["engineering", "industrial", "manufacturing"],
    "Materials": ["chemicals", "cement", "steel", "materials"],
    "Real Estate": ["real estate", "construction", "property"],
    "Telecommunications": ["telecom", "communication", "network"]
}


def detect_sector(file):

    company = file.replace(".pdf","")
    path = os.path.join(reports_folder, file)

    text = ""

    try:
        doc = fitz.open(path)

        # read first 15 pages only (sector usually mentioned early)
        for page in doc[:15]:
            text += page.get_text()

    except:
        return {"Company": company, "Sector": "Unknown"}

    text = text.lower()

    detected_sector = "Unknown"

    for sector, keywords in sector_keywords.items():
        for keyword in keywords:
            if keyword in text:
                detected_sector = sector
                break
        if detected_sector != "Unknown":
            break

    return {"Company": company, "Sector": detected_sector}


if __name__ == "__main__":

    files = [f for f in os.listdir(reports_folder) if f.endswith(".pdf")]

    print("Total reports:", len(files))

    with Pool(cpu_count()) as pool:
        results = pool.map(detect_sector, files)

    df = pd.DataFrame(results)

    # Save company-sector mapping
    df.to_csv("company_sector_mapping_Ann_22_23.csv", index=False)

    # Create sector-wise grouping
    sector_group = df.groupby("Sector")["Company"].apply(list).reset_index()

    sector_group["Companies"] = sector_group["Company"].apply(lambda x: ", ".join(x))

    sector_group = sector_group[["Sector", "Companies"]]

    sector_group.to_csv("sector_wise_companies_Ann_22_23.csv", index=False)

    print("Sector files created successfully")

Total reports: 1394
MuPDF error: argument error: cannot create appearance stream for  widgets

MuPDF error: argument error: cannot create appearance stream for  widgets

MuPDF error: argument error: cannot create appearance stream for  widgets

MuPDF error: argument error: cannot create appearance stream for  widgets

MuPDF error: argument error: cannot create appearance stream for  widgets

MuPDF error: argument error: cannot create appearance stream for  widgets

MuPDF error: argument error: cannot create appearance stream for  widgets

MuPDF error: argument error: cannot create appearance stream for  widgets

MuPDF error: argument error: cannot create appearance stream for  widgets

MuPDF error: argument error: cannot create appearance stream for  widgets

MuPDF error: argument error: cannot create appearance stream for  widgets

MuPDF error: argument error: cannot create appearance stream for  widgets

MuPDF error: argument error: cannot create appearance stream for  widgets

MuPDF

In [4]:
import os
import fitz
import pandas as pd
import nltk
import re
from multiprocessing import Pool, cpu_count
from nltk.corpus import stopwords
import warnings

warnings.filterwarnings("ignore")

nltk.download("stopwords", quiet=True)
stop_words = set(stopwords.words("english"))

reports_folder = "/kaggle/input/datasets/meghanakadari/indian-listed-companies-brsr-reports-fy-202324/Indian_BRSR_Reports_2024-25/Indian_BRSR_Reports_2024-25/BRSR_Reports_2024_25"

sections = [
    "general disclosures",
    "management and process disclosures",
    "principle wise performance",
    "environment",
    "social",
    "governance"
]


def process_pdf(file):

    company = file.replace(".pdf", "")
    path = os.path.join(reports_folder, file)

    text = ""

    try:
        doc = fitz.open(path)

        # read only first 60 pages (BRSR usually early in report)
        for page in doc[:60]:
            text += page.get_text()

    except:
        return None

    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)

    words = text.split()

    total_words = len(words)

    words_no_stop = [w for w in words if w not in stop_words]

    total_words_no_stop = len(words_no_stop)

    results = []

    results.append({
        "Company": company,
        "Section": "TOTAL_REPORT",
        "Word_Count": total_words,
        "Word_Count_No_Stopwords": total_words_no_stop
    })

    for section in sections:

        if section in text:

            section_text = text.split(section, 1)[1]

            words = section_text.split()

            wc = len(words)

            words_no_stop = [w for w in words if w not in stop_words]

            wc_no_stop = len(words_no_stop)

            results.append({
                "Company": company,
                "Section": section,
                "Word_Count": wc,
                "Word_Count_No_Stopwords": wc_no_stop
            })

    return results


if __name__ == "__main__":

    files = [f for f in os.listdir(reports_folder) if f.endswith(".pdf")]

    print("Total reports:", len(files))

    with Pool(cpu_count()) as pool:
        results = pool.map(process_pdf, files)

    data = []

    for r in results:
        if r:
            data.extend(r)

    df = pd.DataFrame(data)

    section_stats = df.groupby("Section")[["Word_Count","Word_Count_No_Stopwords"]].sum()

    company_stats = df[df["Section"]=="TOTAL_REPORT"][["Company","Word_Count","Word_Count_No_Stopwords"]]

    section_presence = pd.crosstab(df["Company"], df["Section"])

    df.to_csv("detailed_word_counts_brsr_24_25.csv", index=False)
    section_stats.to_csv("section_summary_brsr_24_25.csv")
    company_stats.to_csv("company_summary_brsr_24_25.csv")
    section_presence.to_csv("section_presence_brsr_24_25.csv")

    print("EDA Completed")

Total reports: 1167
EDA Completed


In [5]:
import os
import fitz
import pandas as pd
import re
from multiprocessing import Pool, cpu_count

reports_folder = "/kaggle/input/datasets/meghanakadari/indian-listed-companies-brsr-reports-fy-202324/Indian_BRSR_Reports_2024-25/Indian_BRSR_Reports_2024-25/BRSR_Reports_2024_25"

# Sector keyword dictionary
sector_keywords = {
    "Information Technology": ["software", "technology", "it services", "digital"],
    "Financial Services": ["bank", "finance", "financial services", "nbfc", "insurance"],
    "Energy": ["oil", "gas", "energy", "petroleum"],
    "Healthcare": ["pharma", "pharmaceutical", "healthcare", "biotech"],
    "Consumer Goods": ["consumer goods", "fmcg", "retail", "food"],
    "Industrials": ["engineering", "industrial", "manufacturing"],
    "Materials": ["chemicals", "cement", "steel", "materials"],
    "Real Estate": ["real estate", "construction", "property"],
    "Telecommunications": ["telecom", "communication", "network"]
}


def detect_sector(file):

    company = file.replace(".pdf","")
    path = os.path.join(reports_folder, file)

    text = ""

    try:
        doc = fitz.open(path)

        # read first 15 pages only (sector usually mentioned early)
        for page in doc[:15]:
            text += page.get_text()

    except:
        return {"Company": company, "Sector": "Unknown"}

    text = text.lower()

    detected_sector = "Unknown"

    for sector, keywords in sector_keywords.items():
        for keyword in keywords:
            if keyword in text:
                detected_sector = sector
                break
        if detected_sector != "Unknown":
            break

    return {"Company": company, "Sector": detected_sector}


if __name__ == "__main__":

    files = [f for f in os.listdir(reports_folder) if f.endswith(".pdf")]

    print("Total reports:", len(files))

    with Pool(cpu_count()) as pool:
        results = pool.map(detect_sector, files)

    df = pd.DataFrame(results)

    # Save company-sector mapping
    df.to_csv("company_sector_mapping_brsr_24_25.csv", index=False)

    # Create sector-wise grouping
    sector_group = df.groupby("Sector")["Company"].apply(list).reset_index()

    sector_group["Companies"] = sector_group["Company"].apply(lambda x: ", ".join(x))

    sector_group = sector_group[["Sector", "Companies"]]

    sector_group.to_csv("sector_wise_companies_brsr_24_25.csv", index=False)

    print("Sector files created successfully")

Total reports: 1167
Sector files created successfully


In [6]:
import os
import fitz
import pandas as pd
import nltk
import re
from multiprocessing import Pool, cpu_count
from nltk.corpus import stopwords
import warnings

warnings.filterwarnings("ignore")

nltk.download("stopwords", quiet=True)
stop_words = set(stopwords.words("english"))

reports_folder = "/kaggle/input/datasets/meghanakadari/indian-listed-companies-brsr-reports-fy-202324/Indian_BRSR_Reports_2025-26/brsr reports 2025-26/files"

sections = [
    "general disclosures",
    "management and process disclosures",
    "principle wise performance",
    "environment",
    "social",
    "governance"
]


def process_pdf(file):

    company = file.replace(".pdf", "")
    path = os.path.join(reports_folder, file)

    text = ""

    try:
        doc = fitz.open(path)

        # read only first 60 pages (BRSR usually early in report)
        for page in doc[:60]:
            text += page.get_text()

    except:
        return None

    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)

    words = text.split()

    total_words = len(words)

    words_no_stop = [w for w in words if w not in stop_words]

    total_words_no_stop = len(words_no_stop)

    results = []

    results.append({
        "Company": company,
        "Section": "TOTAL_REPORT",
        "Word_Count": total_words,
        "Word_Count_No_Stopwords": total_words_no_stop
    })

    for section in sections:

        if section in text:

            section_text = text.split(section, 1)[1]

            words = section_text.split()

            wc = len(words)

            words_no_stop = [w for w in words if w not in stop_words]

            wc_no_stop = len(words_no_stop)

            results.append({
                "Company": company,
                "Section": section,
                "Word_Count": wc,
                "Word_Count_No_Stopwords": wc_no_stop
            })

    return results


if __name__ == "__main__":

    files = [f for f in os.listdir(reports_folder) if f.endswith(".pdf")]

    print("Total reports:", len(files))

    with Pool(cpu_count()) as pool:
        results = pool.map(process_pdf, files)

    data = []

    for r in results:
        if r:
            data.extend(r)

    df = pd.DataFrame(data)

    section_stats = df.groupby("Section")[["Word_Count","Word_Count_No_Stopwords"]].sum()

    company_stats = df[df["Section"]=="TOTAL_REPORT"][["Company","Word_Count","Word_Count_No_Stopwords"]]

    section_presence = pd.crosstab(df["Company"], df["Section"])

    df.to_csv("detailed_word_counts_brsr_25_26.csv", index=False)
    section_stats.to_csv("section_summary_brsr_25_26.csv")
    company_stats.to_csv("company_summary_brsr_25_26.csv")
    section_presence.to_csv("section_presence_brsr_25_26.csv")

    print("EDA Completed")

Total reports: 1067
EDA Completed


In [7]:
import os
import fitz
import pandas as pd
import re
from multiprocessing import Pool, cpu_count

reports_folder = "/kaggle/input/datasets/meghanakadari/indian-listed-companies-brsr-reports-fy-202324/Indian_BRSR_Reports_2025-26/brsr reports 2025-26/files"

# Sector keyword dictionary
sector_keywords = {
    "Information Technology": ["software", "technology", "it services", "digital"],
    "Financial Services": ["bank", "finance", "financial services", "nbfc", "insurance"],
    "Energy": ["oil", "gas", "energy", "petroleum"],
    "Healthcare": ["pharma", "pharmaceutical", "healthcare", "biotech"],
    "Consumer Goods": ["consumer goods", "fmcg", "retail", "food"],
    "Industrials": ["engineering", "industrial", "manufacturing"],
    "Materials": ["chemicals", "cement", "steel", "materials"],
    "Real Estate": ["real estate", "construction", "property"],
    "Telecommunications": ["telecom", "communication", "network"]
}


def detect_sector(file):

    company = file.replace(".pdf","")
    path = os.path.join(reports_folder, file)

    text = ""

    try:
        doc = fitz.open(path)

        # read first 15 pages only (sector usually mentioned early)
        for page in doc[:15]:
            text += page.get_text()

    except:
        return {"Company": company, "Sector": "Unknown"}

    text = text.lower()

    detected_sector = "Unknown"

    for sector, keywords in sector_keywords.items():
        for keyword in keywords:
            if keyword in text:
                detected_sector = sector
                break
        if detected_sector != "Unknown":
            break

    return {"Company": company, "Sector": detected_sector}


if __name__ == "__main__":

    files = [f for f in os.listdir(reports_folder) if f.endswith(".pdf")]

    print("Total reports:", len(files))

    with Pool(cpu_count()) as pool:
        results = pool.map(detect_sector, files)

    df = pd.DataFrame(results)

    # Save company-sector mapping
    df.to_csv("company_sector_mapping_brsr_25_26.csv", index=False)

    # Create sector-wise grouping
    sector_group = df.groupby("Sector")["Company"].apply(list).reset_index()

    sector_group["Companies"] = sector_group["Company"].apply(lambda x: ", ".join(x))

    sector_group = sector_group[["Sector", "Companies"]]

    sector_group.to_csv("sector_wise_companies_brsr_25_26.csv", index=False)

    print("Sector files created successfully")

Total reports: 1067
Sector files created successfully


In [8]:
#to get the MD&A section (json) from annual reports

!pip install tqdm pdfplumber

import os
import re
import json
import pdfplumber
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

# -------- CONFIG --------
PDF_FOLDER = "/kaggle/input/datasets/vaibhavmeena23/demo-dataset"
OUTPUT_FILE = "mda_dataset.json"
MAX_WORKERS = 4   # adjust based on CPU (4–8 is safe)

# -------- TEXT EXTRACTION --------
def extract_text_from_pdf(pdf_path):
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
    return text.lower()

# -------- CLEAN TEXT --------
def clean_text(text):
    if not text:
        return ""

    text = re.sub(r'[\x00-\x1F\x7F]', ' ', text)

    # Fix merging issues
    text = re.sub(r'([a-z])([A-Z])', r'\1 \2', text)
    text = re.sub(r'([a-zA-Z])(\d)', r'\1 \2', text)
    text = re.sub(r'(\d)([a-zA-Z])', r'\1 \2', text)

    text = re.sub(r'\s+', ' ', text)

    return text.strip()

# -------- MD&A EXTRACTION --------
def find_mda_section(text):
    start_patterns = [
        r"management\s+discussion\s+and\s+analysis",
        r"md\s*&\s*a",
        r"management.?s\s+discussion\s+and\s+analysis",
        r"operating\s+and\s+financial\s+review",
        r"management\s+report"
    ]

    start_idx = None
    for pattern in start_patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            start_idx = match.start()
            break

    if start_idx is None:
        return None

    chunk = text[start_idx:]

    end_patterns = [
        r"standalone financial statements",
        r"consolidated financial statements",
        r"independent auditor",
        r"auditor.?s report",
        r"balance sheet"
    ]

    end_idx = None
    for pattern in end_patterns:
        match = re.search(pattern, chunk, re.IGNORECASE)
        if match:
            end_idx = match.start()
            break

    if end_idx:
        return chunk[:end_idx].strip()
    else:
        return chunk.strip()

# -------- PROCESS SINGLE PDF --------
def process_pdf(file):
    try:
        path = os.path.join(PDF_FOLDER, file)

        text = extract_text_from_pdf(path)
        mda_text = find_mda_section(text)

        if mda_text:
            mda_text = clean_text(mda_text)
        else:
            mda_text = "NOT FOUND"

        return {
            "file": file,
            "mda_text": mda_text
        }

    except Exception as e:
        return {
            "file": file,
            "mda_text": f"ERROR: {str(e)}"
        }

# -------- MAIN PIPELINE --------
def main():
    files = [f for f in os.listdir(PDF_FOLDER) if f.endswith(".pdf")]
    results = []

    with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [executor.submit(process_pdf, f) for f in files]

        for future in tqdm(as_completed(futures), total=len(futures), desc="Processing PDFs"):
            results.append(future.result())

    # Save JSON
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

    print(f"\n✅ Done! Saved to {OUTPUT_FILE}")

# -------- RUN --------
if __name__ == "__main__":
    main()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 61.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 68.2 MB/s eta 0:00:00


Processing PDFs: 100%|██████████| 4/4 [01:43<00:00, 26.00s/it]



✅ Done! Saved to mda_dataset.json


In [9]:
#to get the ESG section (json) from annual reports

import os
import re
import json
import pdfplumber
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

# -------- CONFIG --------
PDF_FOLDER = "/kaggle/input/datasets/vaibhavmeena23/demo-dataset"
OUTPUT_FILE = "esg_dataset.json"
MAX_WORKERS = 4

# -------- TEXT EXTRACTION --------
def extract_text_from_pdf(pdf_path):
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
    return text.lower()

# -------- CLEAN TEXT --------
def clean_text(text):
    if not text:
        return ""

    text = re.sub(r'[\x00-\x1F\x7F]', ' ', text)

    # Fix merging issues
    text = re.sub(r'([a-z])([A-Z])', r'\1 \2', text)
    text = re.sub(r'([a-zA-Z])(\d)', r'\1 \2', text)
    text = re.sub(r'(\d)([a-zA-Z])', r'\1 \2', text)

    text = re.sub(r'\s+', ' ', text)

    return text.strip()

# -------- ESG EXTRACTION --------
def find_esg_section(text):
    # ---- START PATTERNS (very broad) ----
    start_patterns = [
        r"environmental[, ]+social[, ]+and[, ]+governance",
        r"\besg\b",
        r"sustainability",
        r"business responsibility and sustainability report",
        r"\bbrsr\b",
        r"corporate social responsibility",
        r"\bcsr\b",
        r"environmental performance",
        r"social impact",
        r"governance report"
    ]

    matches = []

    # Find ALL ESG-related occurrences
    for pattern in start_patterns:
        for match in re.finditer(pattern, text, re.IGNORECASE):
            matches.append(match.start())

    if not matches:
        return None

    # Start from FIRST ESG mention
    start_idx = min(matches)

    # Take LARGE chunk (high recall)
    chunk = text[start_idx:]

    # ---- END PATTERNS (only strong financial signals) ----
    end_patterns = [
        r"standalone financial statements",
        r"consolidated financial statements",
        r"independent auditor",
        r"auditor.?s report",
        r"balance sheet"
    ]

    end_idx = None
    for pattern in end_patterns:
        match = re.search(pattern, chunk, re.IGNORECASE)
        if match:
            end_idx = match.start()
            break

    if end_idx:
        return chunk[:end_idx].strip()
    else:
        return chunk.strip()


# -------- PROCESS SINGLE PDF --------
def process_pdf(file):
    try:
        path = os.path.join(PDF_FOLDER, file)

        text = extract_text_from_pdf(path)
        esg_text = find_esg_section(text)

        if esg_text:
            esg_text = clean_text(esg_text)
        else:
            esg_text = "NOT FOUND"

        return {
            "file": file,
            "esg_text": esg_text
        }

    except Exception as e:
        return {
            "file": file,
            "esg_text": f"ERROR: {str(e)}"
        }


# -------- MAIN PIPELINE --------
def main():
    files = [f for f in os.listdir(PDF_FOLDER) if f.endswith(".pdf")]
    results = []

    with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [executor.submit(process_pdf, f) for f in files]

        for future in tqdm(as_completed(futures), total=len(futures), desc="Processing ESG"):
            results.append(future.result())

    # Save JSON
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

    print(f"\n✅ Done! Saved to {OUTPUT_FILE}")


# -------- RUN --------
if __name__ == "__main__":
    main()

Processing ESG: 100%|██████████| 4/4 [01:37<00:00, 24.30s/it]



✅ Done! Saved to esg_dataset.json
